In [1]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Initialize a Pinecone client with your API key
api_key = os.environ.get("PINECONE_API_KEY")

In [3]:
pdf_path = "D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\sample_rent.pdf"
pdf_path1 = "D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\dushyant (1).pdf"

In [4]:
# Load PDF
loader = PyPDFLoader(pdf_path)

documents = loader.load()

In [5]:
documents

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2015-11-24T10:05:04-05:00', 'author': 'Federal Trade Commission', 'moddate': '2015-11-24T10:05:04-05:00', 'title': 'Consumer.gov Lesson Materials: Renting an Apartment or House', 'trapped': '/False', 'source': 'D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\sample_rent.pdf', 'total_pages': 3, 'page': 0, 'page_label': '21'}, page_content='Teacher Resources for Consumer.gov | Developed for the FTC by the Center for Applied Linguistics \nSAMPLE RENTAL AGREEMENT\n(Basic / Beginning)\nTHIS AGREEMENT made this 15th Day of June, 2012, by and between ABC Properties, herein called \n“Landlord,” and Silvia Mando, herein called “Tenant.” Landlord hereby agrees to rent to Tenant the dwelling \nlocated at 9876 Cherry Avenue, Apartment 426 under the following terms and conditions.\n1. FIXED-TERM AGREEMENT (LEASE):\nTenants agree to lease this dwelling for a fixed t

### Chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Hierarchy of separators
    is_separator_regex=False,
)

chunks = recursive_splitter.split_documents(documents)

print(len(chunks))
print(chunks[0].page_content)

20
Teacher Resources for Consumer.gov | Developed for the FTC by the Center for Applied Linguistics 
SAMPLE RENTAL AGREEMENT
(Basic / Beginning)
THIS AGREEMENT made this 15th Day of June, 2012, by and between ABC Properties, herein called 
“Landlord,” and Silvia Mando, herein called “Tenant.” Landlord hereby agrees to rent to Tenant the dwelling 
located at 9876 Cherry Avenue, Apartment 426 under the following terms and conditions.
1. FIXED-TERM AGREEMENT (LEASE):


In [7]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2015-11-24T10:05:04-05:00', 'author': 'Federal Trade Commission', 'moddate': '2015-11-24T10:05:04-05:00', 'title': 'Consumer.gov Lesson Materials: Renting an Apartment or House', 'trapped': '/False', 'source': 'D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\sample_rent.pdf', 'total_pages': 3, 'page': 0, 'page_label': '21'}, page_content='Teacher Resources for Consumer.gov | Developed for the FTC by the Center for Applied Linguistics \nSAMPLE RENTAL AGREEMENT\n(Basic / Beginning)\nTHIS AGREEMENT made this 15th Day of June, 2012, by and between ABC Properties, herein called \n“Landlord,” and Silvia Mando, herein called “Tenant.” Landlord hereby agrees to rent to Tenant the dwelling \nlocated at 9876 Cherry Avenue, Apartment 426 under the following terms and conditions.\n1. FIXED-TERM AGREEMENT (LEASE):'),
 Document(metadata={'producer': 'Adobe PDF Libra

#### Storing in Vector Database

In [8]:
# Import the Pinecone library
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

In [19]:
# Create a dense index with integrated embedding

index_name = "legal-lense-index"  ## index name should be unique across your Pinecone project, and name doesn't have _ should be properly formatted

if not pc.has_index(name=index_name):
    pc.create_index(
    name=index_name,
    dimension=1536,  # IMPORTANT
    metric="cosine",
    spec={
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    }
)

##### Mistakes

1. Follow the instructions for the index naming
2. Every Vector DB have certain format of data structure like that.
3. "Vector dimension 1536 does not match the dimension of the index 1024" 
👉 This means:

Your embedding model → produces vectors of size 1536
Your Pinecone index → expects vectors of size 1024

⚠️ These must match exactly, otherwise Pinecone rejects the data.

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [21]:
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
#pc.delete_index(index_name)  

#### Retriever

In [15]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [28]:


# 1. Same embedding model (VERY IMPORTANT)
#embedding_model = OpenAIEmbeddings()
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# 2. Connect to existing index
vectorstore = PineconeVectorStore(
    index_name="legal-lense-index",   # your existing index
    embedding=embeddings
)

# 3. Create retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Maximal Marginal Relevance for diverse results
    search_kwargs={"k": 5}
)



In [29]:
# 4. Query
query = "What is the rent and security deposit amount?"

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

2. RENT:
Tenant agrees to pay Landlord as base rent the sum of $685 per month, due and payable monthly in 
advance on the 1st day of each month during the term of this agreement. The first month’s rent is required 
to be submitted on or before move-in.
3. FORM OF PAYMENT:
Tenants agree to pay their rent in the form of a personal check, a cashier’s check, or a money order made 
out to the Landlord.
4. RENT PAYMENT PROCEDURE:
will it be applied to back or future rent. It will be held intact by Landlord until at least thirty (30) working 
days after Tenants have vacated the property. At that time Landlord will inspect the premises thoroughly 
and assess any damages and/or needed repairs. This deposit money minus any necessary charges for 
missing/dead light bulbs, repairs, cleaning, etc., will then be returned to Tenant with a written explanation 
of deductions, within 60 days after they have vacated the property.
to serve the handicapped, such as seeing-eye dogs, hearing dogs, or service

In [30]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [31]:
# 5. Prompt
prompt = ChatPromptTemplate.from_template("""
You are a legal assistant.

Answer ONLY from the context below.
If not found, say "I don't know".

Context:
{context}

Question:
{question}
""")

In [32]:
# 6. Convert docs → text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7. Chain (LCEL)
chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [34]:
# 8. Run
response = chain.invoke("What is the rent amount? of property")
print(response)

The rent amount is $685 per month.


### Failed at - Rent and security deposit amount - simple vector

In [ ]:
# 3. Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",   # basic vector search
    search_kwargs={"k": 3}      # top 3 results
)

# 4. Query
query = "What is bail law in India?"

docs = retriever.invoke(query)

# 5. Print results
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

In [21]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

In [22]:
query = "What is the notice period?"

docs = vectorstore.similarity_search(query, k=3)

for doc in docs:
    print(doc.page_content)
    print("-----")

will it be applied to back or future rent. It will be held intact by Landlord until at least thirty (30) working 
days after Tenants have vacated the property. At that time Landlord will inspect the premises thoroughly 
and assess any damages and/or needed repairs. This deposit money minus any necessary charges for 
missing/dead light bulbs, repairs, cleaning, etc., will then be returned to Tenant with a written explanation 
of deductions, within 60 days after they have vacated the property.
-----
12. NOTIFICATION OF SERIOUS BUILDING PROBLEMS:
Tenant agrees to notify Landlord immediately if roof leaks, water spots appear on ceiling, or at the first sign 
of termite activity. Tenants also agree to notify the Owners immediately upon first discovering any signs of 
serious building problems such as foundation cracks, a tilting porch, a crack in plaster, buckling drywall or 
siding, a spongy floor, a leaky water heater, etc. If the tenant does not notify landlord in a prompt matter the
---

### Evaluation

In [33]:
import pandas as pd

df_eval = pd.read_excel("D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\rag_eval_dataset.xlsx")

questions = df_eval["question"].tolist()
ground_truths = df_eval["ground_truth"].tolist()

In [34]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

embedding_model = OpenAIEmbeddings()

vectorstore = PineconeVectorStore(
    index_name="legal-lense-index",
    embedding=embedding_model
)



llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    n=3   # allow multiple generations
)

In [25]:
### create retriver here
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Maximal Marginal Relevance for diverse results
    search_kwargs={"k": 5}
)

In [26]:
answers = []
contexts = []

for query in questions:
    # Retrieve docs
    docs = retriever.invoke(query)
    
    # Store context
    context_texts = [doc.page_content for doc in docs]
    contexts.append(context_texts)
    
    # Create prompt manually (LCEL style)
    context_str = "\n\n".join(context_texts)
    
    prompt = f"""
    Answer ONLY from the context below.
    
    Context:
    {context_str}
    
    Question:
    {query}
    """
    
    response = llm.invoke(prompt)
    
    answers.append(response.content)

In [27]:
from datasets import Dataset

data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(data)

In [28]:
dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})

In [29]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
from ragas.embeddings import OpenAIEmbeddings

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

from ragas.llms import llm_factory

ragas_llm = llm_factory(
    "gpt-4o-mini",
    client=client
)

# 2. Embeddings – use the explicit class
ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=client,          # reuse the same OpenAI client
)

# ✅ Correct imports for Ragas 0.4.3
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas import evaluate   # <-- also add this import

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
]

# Make sure `dataset` is defined before this line
result = evaluate(
    dataset,
    metrics=metrics,
)

print(result)

C:\Users\Oliver\AppData\Local\Temp\ipykernel_26948\2746582949.py:25: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_26948\2746582949.py:25: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_26948\2746582949.py:25: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 0.0500, 'context_recall': 0.2000}


#### Cohere Reranking

In [9]:
import pandas as pd

df_eval = pd.read_excel("D:\\Data_analytics_new\\Sub_Resume_Building\\Legal_Lense\\data\\rag_eval_dataset.xlsx")

questions = df_eval["question"].tolist()
ground_truths = df_eval["ground_truth"].tolist()

In [10]:
from dotenv import load_dotenv
load_dotenv()
import os
import cohere
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")

# LangChain setup
embedding_model = OpenAIEmbeddings()

vectorstore = PineconeVectorStore(
    index_name="legal-lense-index",
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Cohere client
co = cohere.Client()


def rerank_documents(query, docs, top_n=3):
    doc_texts = [doc.page_content for doc in docs]

    response = co.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=doc_texts,
        top_n=top_n
    )

    return [docs[r.index] for r in response.results]


def rag_with_rerank(query):
    # Retrieve
    docs = retriever.invoke(query)

    # Rerank
    reranked_docs = rerank_documents(query, docs, top_n=3)

    # Context
    context = "\n\n".join([doc.page_content for doc in reranked_docs])

    # Prompt
    prompt = f"""
    You are a legal assistant.
    Answer ONLY from the context below.
    If not found, say "I don't know".

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    return response.content, reranked_docs

In [12]:
answers = []
contexts = []

import time

for q in questions:
    answer, docs = rag_with_rerank(q)

    answers.append(answer)
    contexts.append([doc.page_content for doc in docs])

    time.sleep(7)  # 🔥 ~8-9 calls per minute (safe)

In [13]:
from datasets import Dataset

eval_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(eval_data)

In [14]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas import evaluate

client = OpenAI()

ragas_llm = llm_factory(
    "gpt-4o-mini",
    client=client
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=client,
)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
]

result = evaluate(
    dataset,
    metrics=metrics,
)

print(result)

C:\Users\Oliver\AppData\Local\Temp\ipykernel_12380\1373804146.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_12380\1373804146.py:8: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_12380\1373804146.py:8: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

{'faithfulness': 0.6250, 'answer_relevancy': nan, 'context_precision': 0.6167, 'context_recall': 0.5750}


#### Hybrid Search

In [35]:
from langchain_community.retrievers import BM25Retriever

# Create BM25 from your chunks
bm25_retriever = BM25Retriever.from_documents(chunks)

# Set how many docs BM25 returns
bm25_retriever.k = 10

In [36]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [37]:
from collections import defaultdict

def hybrid_retrieve(query, top_k=5):
    # Step 1: BM25 raw scores
    bm25_results = bm25_retriever.invoke(query)
    bm25_scores_raw = bm25_retriever.vectorizer.get_scores(query.split())

    # Step 2: Vector similarity scores
    vector_results = vectorstore.similarity_search_with_score(query, k=10)

    # Convert to dict
    vector_scores_raw = {doc.page_content: score for doc, score in vector_results}

    # Step 3: Normalize scores
    def normalize(scores):
        min_s = min(scores)
        max_s = max(scores)
        return [(s - min_s) / (max_s - min_s + 1e-8) for s in scores]

    # Normalize BM25
    bm25_norm_scores = normalize(bm25_scores_raw)

    # Map BM25 scores to docs
    bm25_scores = {}
    for doc, score in zip(chunks, bm25_norm_scores):
        bm25_scores[doc.page_content] = score

    # Normalize vector scores (lower distance = better)
    if vector_scores_raw:
        vec_values = list(vector_scores_raw.values())
        vec_norm = normalize([-v for v in vec_values])  # invert distance
        vector_scores = dict(zip(vector_scores_raw.keys(), vec_norm))
    else:
        vector_scores = {}

    # Step 4: Combine scores
    combined = defaultdict(float)

    for text, score in bm25_scores.items():
        combined[text] += 0.4 * score

    for text, score in vector_scores.items():
        combined[text] += 0.6 * score

    # Step 5: Sort
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    # Step 6: Map back to documents
    doc_map = {doc.page_content: doc for doc in chunks}

    return [doc_map[text] for text, _ in ranked[:top_k]]

In [38]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def hybrid_rag(query):
    docs = hybrid_retrieve(query, top_k=3)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    You are a strict legal assistant.

    You MUST follow these rules:

    1. Answer ONLY using the provided context.
    2. Do NOT use any external knowledge.
    3. Do NOT make assumptions or infer missing information.
    4. If the answer is not explicitly stated in the context, respond exactly with:
    "I don't know."
    5. Do NOT paraphrase beyond what is supported by the context.
    6. If multiple pieces of information exist, combine them ONLY if clearly supported.

    ---

    Context:
    {context}

    ---

    Question:
    {query}

    ---

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content, docs

In [39]:
questions

['What is the duration of the lease agreement?',
 'What happens after the lease term expires?',
 'What is the monthly rent amount?',
 'When is the rent due each month?',
 'Is there a grace period for rent payment?',
 'What is the late fee after the grace period?',
 'What forms of payment are accepted for rent?',
 'Where should rent payments be sent?',
 'What happens if a tenant’s check bounces?',
 'How much is the security deposit?',
 'When is the security deposit returned?',
 'Can the security deposit be used as rent?',
 'What is the cleaning fee if the property is not clean?',
 'How many vehicles are tenants allowed?',
 'Who pays for utilities?',
 'What should tenants do if there are serious building issues?',
 'Are pets allowed in the property?',
 'Is there an additional charge for pets?',
 'What happens if landlord property is removed without permission?',
 'What does full disclosure mean in this agreement?']

In [40]:
test_queries = questions

answers = []
contexts = []

for q in test_queries:
    ans, docs = hybrid_rag(q)

    answers.append(ans)
    contexts.append([doc.page_content for doc in docs])

In [41]:
from datasets import Dataset

eval_data = {
    "question": test_queries,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(eval_data)

In [51]:
test_queries[3]

'When is the rent due each month?'

In [52]:
contexts[3]

['2. RENT:\nTenant agrees to pay Landlord as base rent the sum of $685 per month, due and payable monthly in \nadvance on the 1st day of each month during the term of this agreement. The first month’s rent is required \nto be submitted on or before move-in.\n3. FORM OF PAYMENT:\nTenants agree to pay their rent in the form of a personal check, a cashier’s check, or a money order made \nout to the Landlord.\n4. RENT PAYMENT PROCEDURE:',
 'Teacher Resources for Consumer.gov | Developed for the FTC by the Center for Applied Linguistics \nSAMPLE RENTAL AGREEMENT\n(Basic / Beginning)\nTHIS AGREEMENT made this 15th Day of June, 2012, by and between ABC Properties, herein called \n“Landlord,” and Silvia Mando, herein called “Tenant.” Landlord hereby agrees to rent to Tenant the dwelling \nlocated at 9876 Cherry Avenue, Apartment 426 under the following terms and conditions.\n1. FIXED-TERM AGREEMENT (LEASE):',
 'tenant may be held financially responsible.']

In [43]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas import evaluate

client = OpenAI()

ragas_llm = llm_factory(
    "gpt-4o-mini",
    client=client
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=client,
)

metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
]

result = evaluate(
    dataset,
    metrics=metrics,
)

print(result)

C:\Users\Oliver\AppData\Local\Temp\ipykernel_22224\1373804146.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_22224\1373804146.py:8: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\Oliver\AppData\Local\Temp\ipykernel_22224\1373804146.py:8: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

{'faithfulness': 0.4500, 'answer_relevancy': nan, 'context_precision': 0.3750, 'context_recall': 0.3750}
